In [1]:
# 1) 한 번만: 압축 풀기
!tar xzf sample_data/EnglishFnt.tgz -C sample_data/

In [2]:
# ────────────────────────────────────────────────────────────────
# 0) EnglishFnt에서 D/N/E 데이터셋 자동 구축
# ────────────────────────────────────────────────────────────────
import os
from pathlib import Path
from PIL import Image
import shutil
import random

# 원본 폰트 이미지 경로
FNT_BASE = Path("sample_data/English/Fnt")
OUT_BASE = Path("dataset")
CLASS_MAP = {'Sample014': 'D', 'Sample024': 'N', 'Sample015': 'E'}

# dataset/D, dataset/N, dataset/E 폴더 생성 및 초기화
for c in ['D', 'N', 'E']:
    d = OUT_BASE / c
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

# 클래스별로 파일 복사 (최대 2000장씩 샘플)
for sample, label in CLASS_MAP.items():
    src = FNT_BASE / sample
    dst = OUT_BASE / label
    files = list(src.glob("*.png"))
    random.shuffle(files)
    for i, f in enumerate(files[:2000]):
        img = Image.open(f).convert("L").resize((32,32))
        img.save(dst / f"{label}_{i:04d}.png")

print("✅ EnglishFnt → dataset/D,N,E 자동 구축 완료")

# ────────────────────────────────────────────────────────────────
# 1) Keras 데이터셋 로드 및 증강
# ────────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE    = (32, 32)
BATCH_SIZE  = 64
EPOCHS      = 20
TFLITE_PATH = "dne_classifier.tflite"
CLASS_NAMES = ["D", "N", "E"]
VALID_SPLIT = 0.2
SEED        = 42

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset",
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    validation_split=VALID_SPLIT,
    subset="training",
    seed=SEED
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset",
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    validation_split=VALID_SPLIT,
    subset="validation",
    seed=SEED
)

normalization = layers.Rescaling(1.0 / 255)
data_augmentation = tf.keras.Sequential([
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
    layers.RandomRotation(0.1),
])

def preprocess_train(x, y):
    x = tf.expand_dims(x, -1) if x.shape[-1] != 1 else x
    x = data_augmentation(x)
    x = normalization(x)
    return x, y

train_ds = train_ds.map(preprocess_train).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (normalization(x), y)).prefetch(buffer_size=tf.data.AUTOTUNE)

# ────────────────────────────────────────────────────────────────
# 2) CNN 모델 정의 및 학습
# ────────────────────────────────────────────────────────────────
def build_dne_model(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax")
    ])
    return model

model = build_dne_model(IMG_SIZE + (1,), len(CLASS_NAMES))
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# ────────────────────────────────────────────────────────────────
# 3) TFLite 변환 및 저장
# ────────────────────────────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)
print(f"✅ 3-class TFLite model saved at '{TFLITE_PATH}'")


✅ EnglishFnt → dataset/D,N,E 자동 구축 완료
Found 3048 files belonging to 3 classes.
Using 2439 files for training.
Found 3048 files belonging to 3 classes.
Using 609 files for validation.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 543,619 (2.07 MB)

 Trainable params: 543,619 (2.07 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 8s 86ms/step - accuracy: 0.5501 - loss: 0.9543 - val_accuracy: 0.9573 - val_loss: 0.1312
Epoch 2/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.9239 - loss: 0.2327 - val_accuracy: 0.9688 - val_loss: 0.0784
Epoch 3/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9473 - loss: 0.1399 - val_accuracy: 0.9737 - val_loss: 0.0681
Epoch 4/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9601 - loss: 0.1259 - val_accuracy: 0.9787 - val_loss: 0.0574
Epoch 5/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9518 - loss: 0.1145 - val_accuracy: 0.9836 - val_loss: 0.0527
Epoch 6/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9755 - loss: 0.0809 - val_accuracy: 0.9869 - val_loss: 0.0469
Epoch 7/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9712 - loss: 0.0824 - val_accuracy: 0.9885 - val_loss: 0.0410
Epoch 8/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9795 - loss: 0.0607 - val_accuracy: 0.9918 - v

In [34]:
import cv2
import numpy as np
import json
import tensorflow as tf

# ─────────────────────────────
# 0) 설정
# ─────────────────────────────
IMAGE_PATH   = '/content/schedule4.jpg'      # 이미지 경로
TFLITE_MODEL = 'dne_classifier.tflite'       # 3-class TFLite 모델 경로
WHITE_THRESH = 0.86                          # 빈 셀 판단 임계치
CLASS_LABELS = ['D', 'N', 'E']               # 3-class
HEADER_SKIP_Y = 2   # 상단 헤더(번호+이름) 행 개수
HEADER_SKIP_X = 1   # 왼쪽 헤더(이름) 열 개수
DEBUG_VIS = True    # 표 전체 box 시각화 저장

# ─────────────────────────────
# 1) 라인 감지 + 셀 좌표 추출
# ─────────────────────────────
def get_table_cells(img_path, header_skip_y=2, header_skip_x=1, vis_path="table_detected.png"):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    assert img is not None, f"Image not found at {img_path}"

    # Binarize (invert + Otsu)
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Morphology to extract border lines
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (100, 1))
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 100))
    h_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, h_kernel)
    v_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, v_kernel)

    def find_seps(mask, axis, ratio=0.5):
        proj = mask.sum(axis=0) if axis=='x' else mask.sum(axis=1)
        thresh = proj.max() * ratio
        idx = np.where(proj > thresh)[0]
        groups = np.split(idx, np.where(np.diff(idx) > 1)[0] + 1)
        return sorted(int(np.median(g)) for g in groups if len(g) > 0)

    x_seps = find_seps(v_lines, 'x', 0.5)
    y_seps = find_seps(h_lines, 'y', 0.5)

    # (시각화)
    if vis_path:
        img_vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        for y in y_seps:
            cv2.line(img_vis, (x_seps[0], y), (x_seps[-1], y), (0,0,255), 2)
        for x in x_seps:
            cv2.line(img_vis, (x, y_seps[0]), (x, y_seps[-1]), (0,0,255), 2)
        cv2.imwrite(vis_path, img_vis)
        print(f"Table lines visualization saved: {vis_path}")

    # (좌표 반환)
    data_rows = list(zip(y_seps[header_skip_y-1:-1], y_seps[header_skip_y:]))

    data_cols = list(zip(x_seps[header_skip_x:-1], x_seps[header_skip_x+1:]))
    return img, data_rows, data_cols

# ─────────────────────────────
# 2) 셀 crop + 리사이즈
# ─────────────────────────────
def tight_crop_and_resize(cell_img, out_size=(32,32)):
    if cell_img is None or cell_img.size == 0:
        return np.full(out_size, 255, dtype=np.uint8)
    # cell_img: 2D gray
    _, threshed = cv2.threshold(cell_img, 200, 255, cv2.THRESH_BINARY)
    inv = 255 - threshed
    coords = cv2.findNonZero(inv)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        cropped = cell_img[y:y+h, x:x+w]
    else:
        cropped = cell_img
    resized = cv2.resize(cropped, out_size)
    return resized

# ─────────────────────────────
# 3) 셀 분류 함수
# ─────────────────────────────
def classify_dne(cell_img, interpreter, input_details, output_details, debug=False, r=0, c=0):
    tight_img = tight_crop_and_resize(cell_img)
    white_ratio = np.mean(tight_img > 220)
    if debug:
        cv2.imwrite(f'debug_cell_r{r}_c{c}.png', tight_img)
    if white_ratio > WHITE_THRESH:
        if debug: print("Blank cell detected: white_ratio=", white_ratio)
        return '-'
    h, w = input_details[0]['shape'][1:3]
    inp = tight_img.astype(np.float32) / 255.0
    sample = inp.reshape(1, h, w, 1)
    interpreter.set_tensor(input_details[0]['index'], sample)
    interpreter.invoke()
    out = interpreter.get_tensor(output_details[0]['index'])[0]
    if debug: print(f"Probabilities: D={out[0]:.3f}, N={out[1]:.3f}, E={out[2]:.3f}")
    idx = np.argmax(out)
    label = CLASS_LABELS[idx]
    if debug: print("Predicted label:", label)
    return label

# ─────────────────────────────
# 4) 메인 실행 & JSON 변환
# ─────────────────────────────
def main():
    # (1) 셀 경계 자동 추출
    img, data_rows, data_cols = get_table_cells(
        IMAGE_PATH, header_skip_y=2, header_skip_x=1, vis_path="table_detected.png"
    )

    # (2) TFLite 모델 준비
    interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL)
    interpreter.allocate_tensors()
    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # (3) 셀 순회하며 분류
    result = {}
    debug_count = 0
    for r, (y1, y2) in enumerate(data_rows):
        row_dict = {}
        for c, (x1, x2) in enumerate(data_cols):
            cell_img = img[y1:y2, x1:x2]
            debug = (debug_count < 30)
            # 파일 이름에 좌표도 추가!
            debug = (debug_count < 30)
            val = classify_dne(cell_img, interpreter, input_details, output_details, debug=debug, r=r, c=c)
            row_dict[str(c+1)] = val
            if debug: debug_count += 1
        result[str(r+1)] = row_dict


    json_str = json.dumps(result, ensure_ascii=False, indent=2)
    print(json_str)
    with open('schedule_inferred.json', 'w', encoding='utf-8') as f:
        f.write(json_str)
    print("schedule_inferred.json 생성 완료")

if __name__ == '__main__':
    main()


Table lines visualization saved: table_detected.png
Probabilities: D=1.000, N=0.000, E=0.000
Predicted label: D
Probabilities: D=1.000, N=0.000, E=0.000
Predicted label: D
Blank cell detected: white_ratio= 0.896484375
Blank cell detected: white_ratio= 0.896484375
Probabilities: D=0.000, N=1.000, E=0.000
Predicted label: N
Probabilities: D=0.000, N=1.000, E=0.000
Predicted label: N
Probabilities: D=0.000, N=1.000, E=0.000
Predicted label: N
Probabilities: D=0.000, N=0.000, E=1.000
Predicted label: E
Probabilities: D=0.000, N=0.000, E=1.000
Predicted label: E
Blank cell detected: white_ratio= 0.896484375
Blank cell detected: white_ratio= 0.89453125
Probabilities: D=1.000, N=0.000, E=0.000
Predicted label: D
Probabilities: D=1.000, N=0.000, E=0.000
Predicted label: D
Probabilities: D=0.000, N=1.000, E=0.000
Predicted label: N
Probabilities: D=0.000, N=0.000, E=1.000
Predicted label: E
Blank cell detected: white_ratio= 0.896484375
Probabilities: D=0.000, N=1.000, E=0.000
Predicted label: N